# Loading `test_<frame_no>.txt` folders

This notebook checks `load_test_folder()` against `load_united_txt()` on the overlapping frame range.

The folder export contains all frames, while `unitedc_62_239.txt` contains only frames 62..239, so a whole-folder vs. united-file equality check would be misleading.

In [ ]:
from pathlib import Path

import numpy as np

from pha_lib import io

ROOT = Path(r"d:\ℱ Sci\IFPiLM\1st task")
FRAMES = ROOT / "data" / "test" / "frames"
UNITED = ROOT / "data" / "test" / "modified" / "unitedc.txt"

sample_files = sorted(fp for fp in FRAMES.iterdir() if fp.is_file() and io._TEST_FILE_RE.search(fp.name))[:5]
print("Sample folder files:")
for file_path in sample_files:
    frame_no = io._TEST_FILE_RE.search(file_path.name).group(1)
    print(f"  {file_path.name} -> frame {frame_no}")

shot_folder = io.load_test_folder(FRAMES, discharge_id="frames")
shot_united = io.load_united_txt(UNITED, discharge_id="unitedc_62_239")

print()
print("Loaded folder discharge:")
print(f"  frames: {shot_folder.meta['n_frames']}")
print(f"  bins:   {shot_folder.meta['n_bins']}")
print(f"  range:  {shot_folder.channels[1].frame_numbers[0]}..{shot_folder.channels[1].frame_numbers[-1]}")

print()
print("Loaded united discharge:")
print(f"  frames: {shot_united.meta['n_frames']}")
print(f"  bins:   {shot_united.meta['n_bins']}")
print(f"  range:  {shot_united.channels[1].frame_numbers[0]}..{shot_united.channels[1].frame_numbers[-1]}")

shared_frames = np.intersect1d(
    shot_folder.channels[1].frame_numbers,
    shot_united.channels[1].frame_numbers,
)
folder_mask = np.isin(shot_folder.channels[1].frame_numbers, shared_frames)
united_mask = np.isin(shot_united.channels[1].frame_numbers, shared_frames)

print()
print(f"Shared frames: {shared_frames[0]}..{shared_frames[-1]} ({len(shared_frames)})")

for channel_id in (1, 2):
    folder_channel = shot_folder.channels[channel_id]
    united_channel = shot_united.channels[channel_id]

    np.testing.assert_array_equal(
        folder_channel.frame_numbers[folder_mask],
        united_channel.frame_numbers[united_mask],
    )
    np.testing.assert_allclose(folder_channel.energy_eV[:-1], united_channel.energy_eV)
    np.testing.assert_allclose(folder_channel.spectra[folder_mask, :-1], united_channel.spectra[united_mask])

print("Shared frame range matches, and the first 2047 bins match exactly for both channels.")

Sample folder files:
  test_full_1.txt -> frame 1
  test_full_10.txt -> frame 10
  test_full_100.txt -> frame 100
  test_full_101.txt -> frame 101
  test_full_102.txt -> frame 102

Loaded folder discharge:
  frames: 532
  bins:   2048
  range:  1..532

Loaded united discharge:
  frames: 178
  bins:   2047
  range:  62..239

Shared frames: 62..239 (178)
Shared frame range matches, and the first 2047 bins match exactly for both channels.
